# Demo Modul 5: Convolutional Neural Network Dasar

**Durasi sesi:** 120 menit  
**Kasus:** CIFAR-10 $3\times32\times32$; Fashion-MNIST tersedia sebagai opsi CPU ringan.

Empat modul sebelumnya meratakan citra menjadi vektor. Notebook ini memeriksa apa yang hilang karena kebiasaan itu.

## Capaian demo

Setelah demo, praktikan dapat:

1. membaca tensor $N\times C\times H\times W$ dan menghitung shape keluaran sebelum menjalankan kode;
2. menghitung parameter convolution dan menemukan di mana parameter sebenarnya berkumpul;
3. membandingkan FNN dan CNN pada anggaran parameter yang setara;
4. mengukur dampak perubahan kernel dan jumlah kanal; dan
5. membaca feature map serta kesalahan klasifikasi.

In [ ]:
import platform
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from torchvision.datasets import CIFAR10, FashionMNIST

DATASET = 'cifar10'          # ganti ke 'fashion' bila hanya tersedia CPU
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
pd.set_option('display.precision', 4)
print({'torch': torch.__version__, 'device': str(DEVICE), 'dataset': DATASET})

## 1. Format tensor dan rumus shape

$$n_\text{keluar}=\left\lfloor\frac{n+2p-k}{s}\right\rfloor+1$$

Hitung dahulu di kepala, baru periksa dengan kode.

In [ ]:
def shape_keluar(n, k, p=0, s=1):
    return (n + 2 * p - k) // s + 1

for n, k, p, s in [(32, 3, 1, 1), (32, 3, 0, 1), (32, 5, 2, 1), (32, 2, 0, 2)]:
    print(f'n={n} k={k} p={p} s={s}  ->  {shape_keluar(n, k, p, s)}')

conv = nn.Conv2d(3, 32, kernel_size=3, padding=1)
x = torch.randn(4, 3, 32, 32)                      # N, C, H, W
print('\nmasukan :', tuple(x.shape))
print('keluaran:', tuple(conv(x).shape))
print('parameter conv:', sum(p.numel() for p in conv.parameters()),
      '= 32*3*3*3 + 32')

**Pemeriksaan:** $k=3,p=1,s=1$ mempertahankan ukuran; \; $k=2,s=2$ (pooling) membaginya dua. Perhatikan pula bahwa jumlah parameter convolution **tidak** memuat $32\times32$ — ukuran citra tidak menambah parameter.

## 2. Data: subset tetap

Protokol modul: $10\,000$ citra latih dan $2\,000$ validasi, terstratifikasi. Normalisasi per kanal dari subset latih saja.

In [ ]:
DATA_ROOT = Path('../../data/raw')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('data/raw')

if DATASET == 'cifar10':
    tr = CIFAR10(root=DATA_ROOT, train=True, download=False)
    te = CIFAR10(root=DATA_ROOT, train=False, download=False)
    X_penuh = torch.tensor(tr.data).permute(0, 3, 1, 2).float() / 255.0
    y_penuh = torch.tensor(tr.targets)
    X_uji_mentah = torch.tensor(te.data).permute(0, 3, 1, 2).float() / 255.0
    y_uji = torch.tensor(te.targets)
    KELAS = tr.classes
else:
    tr = FashionMNIST(root=DATA_ROOT, train=True, download=False)
    te = FashionMNIST(root=DATA_ROOT, train=False, download=False)
    X_penuh = tr.data.unsqueeze(1).float() / 255.0
    y_penuh = tr.targets
    X_uji_mentah = te.data.unsqueeze(1).float() / 255.0
    y_uji = te.targets
    KELAS = tr.classes

C_IN, H, W = X_penuh.shape[1:]
idx_latih, idx_val = train_test_split(
    np.arange(len(y_penuh)), train_size=10_000, test_size=2_000,
    stratify=y_penuh.numpy(), random_state=SEED)

X_latih, y_latih = X_penuh[idx_latih], y_penuh[idx_latih]
X_val, y_val = X_penuh[idx_val], y_penuh[idx_val]

MEAN = X_latih.mean(dim=(0, 2, 3), keepdim=True)     # per kanal
STD = X_latih.std(dim=(0, 2, 3), keepdim=True)
normalkan = lambda t: (t - MEAN) / STD

ds_latih = TensorDataset(normalkan(X_latih), y_latih)
ds_val = TensorDataset(normalkan(X_val), y_val)
ds_uji = TensorDataset(normalkan(X_uji_mentah), y_uji)

print(f'bentuk satu citra: {C_IN}x{H}x{W}')
print(f'latih {len(ds_latih)}  validasi {len(ds_val)}  uji {len(ds_uji)}')
print('mean per kanal:', MEAN.flatten().tolist())

## 3. CNN dua blok dan letak parameternya

In [ ]:
def buat_cnn(c1=32, c2=64, k=3, c_in=C_IN, hw=H):
    seed_everything(SEED)
    p = k // 2                                  # menjaga ukuran tetap
    sisi = hw // 4                              # dua kali MaxPool2d(2)
    return nn.Sequential(
        nn.Conv2d(c_in, c1, kernel_size=k, padding=p), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(c1, c2, kernel_size=k, padding=p), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(c2 * sisi * sisi, 128), nn.ReLU(),
        nn.Linear(128, 10),
    ).to(DEVICE)

cnn = buat_cnn()
rincian = [(nama, sum(q.numel() for q in m.parameters()))
           for nama, m in cnn.named_children()
           if sum(q.numel() for q in m.parameters()) > 0]
total = sum(n for _, n in rincian)
for nama, n in rincian:
    print(f'  layer {nama:>2}: {n:>9,} parameter  ({100*n/total:5.1f}%)')
print(f'TOTAL: {total:,}')

**Temuan yang layak direnungkan:** pada CIFAR-10, sekitar **96%** parameter berada pada satu layer `Linear`, bukan pada convolution-nya. Bagian yang mengenali pola justru bagian yang paling murah.

In [ ]:
# Verifikasi shape tiap blok dengan forward hook.
catatan_shape = []
kait = []
for nama, m in cnn.named_children():
    kait.append(m.register_forward_hook(
        lambda mod, inp, out, nm=nama: catatan_shape.append(
            (nm, type(mod).__name__, tuple(out.shape)))))

with torch.no_grad():
    cnn(torch.randn(4, C_IN, H, W, device=DEVICE))
for k in kait:
    k.remove()

pd.DataFrame(catatan_shape, columns=['layer', 'modul', 'keluaran'])

## 4. FNN pembanding pada anggaran setara

Agar selisih kinerja tidak dapat dijelaskan oleh ukuran model, FNN disetel agar jumlah parameternya praktis sama dengan CNN.

In [ ]:
def buat_fnn(hidden, c_in=C_IN, hw=H):
    seed_everything(SEED)
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(c_in * hw * hw, hidden), nn.ReLU(),
        nn.Linear(hidden, 10),
    ).to(DEVICE)

target = sum(p.numel() for p in cnn.parameters())
d = C_IN * H * W
hidden = round((target - 10) / (d + 1 + 10))          # hidden yang menyamakan anggaran
fnn = buat_fnn(hidden)
print(f'hidden FNN  : {hidden}')
print(f'parameter CNN: {target:,}')
print(f'parameter FNN: {sum(p.numel() for p in fnn.parameters()):,}')

In [ ]:
BATCH, EPOCH = 128, 10

def loader(ds, batch, acak):
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(ds, batch_size=batch, shuffle=acak, generator=g if acak else None)

@torch.no_grad()
def evaluasi(model, ds, batch=512):
    model.eval()
    kriteria = nn.CrossEntropyLoss(reduction='sum')
    total, benar = 0.0, 0
    for xb, yb in loader(ds, batch, False):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        total += kriteria(logits, yb).item()
        benar += (logits.argmax(1) == yb).sum().item()
    return total / len(ds), benar / len(ds)

def jalankan(model, label, epoch=EPOCH):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    kriteria = nn.CrossEntropyLoss()
    dl = loader(ds_latih, BATCH, True)
    riwayat, n_update, mulai = {'val_loss': [], 'val_acc': []}, 0, time.perf_counter()

    for _ in range(epoch):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            kriteria(model(xb), yb).backward()
            opt.step()
            n_update += 1
        vl, va = evaluasi(model, ds_val)
        riwayat['val_loss'].append(vl); riwayat['val_acc'].append(va)

    tl, _ = evaluasi(model, ds_latih)
    durasi = time.perf_counter() - mulai
    return riwayat, {
        'run_id': label, 'seed': SEED, 'arsitektur': label,
        'parameter': sum(p.numel() for p in model.parameters()),
        'n_update': n_update, 'train_loss': tl,
        'val_loss': riwayat['val_loss'][-1], 'val_acc': riwayat['val_acc'][-1],
        'gap': riwayat['val_loss'][-1] - tl,
        'detik_per_epoch': durasi / epoch, 'runtime_s': durasi,
    }

hasil, kurva = [], {}
for model, label in [(fnn, 'fnn'), (buat_cnn(), 'cnn-baseline')]:
    r, catatan = jalankan(model, label)
    hasil.append(catatan); kurva[label] = r

pd.DataFrame(hasil)[['run_id', 'parameter', 'n_update', 'val_loss', 'val_acc',
                     'gap', 'detik_per_epoch']]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.4))
for label, r in kurva.items():
    ax.plot(range(1, EPOCH + 1), r['val_acc'], marker='o', label=label)
ax.set_xlabel('epoch'); ax.set_ylabel('validation accuracy')
ax.set_title('FNN dan CNN pada anggaran parameter yang setara')
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

Karena jumlah parameter keduanya hampir sama, selisih ini **tidak** dapat dijelaskan oleh ukuran model. Yang membedakan adalah cara parameter disusun: convolution mempertahankan susunan spasial dan memakai ulang kernel yang sama di seluruh posisi.

## 5. Dua varian terkendali

Satu varian, satu perubahan.

In [ ]:
for label, kwargs in [('varian-A-k5', dict(k=5)),
                      ('varian-B-kanal', dict(c1=64, c2=128))]:
    model = buat_cnn(**kwargs)
    print(f'{label}: {sum(p.numel() for p in model.parameters()):,} parameter')
    _, catatan = jalankan(model, label)
    hasil.append(catatan)

tabel = pd.DataFrame(hasil)
tabel[['run_id', 'parameter', 'val_loss', 'val_acc', 'gap', 'detik_per_epoch']]

Perhatikan varian B: menggandakan kanal blok kedua membuat masukan layer `Linear` ikut berlipat, sehingga total parameternya melonjak jauh melampaui baseline. Inilah alasan arsitektur modern mengganti `Flatten` dengan global average pooling.

## 6. Feature map dan kesalahan klasifikasi

In [ ]:
terbaik = buat_cnn()
_, catatan_terbaik = jalankan(terbaik, 'cnn-untuk-analisis')

contoh = ds_val[0][0].unsqueeze(0).to(DEVICE)
peta = {}
kait = []
for nama, m in terbaik.named_children():
    if isinstance(m, nn.Conv2d):
        kait.append(m.register_forward_hook(
            lambda mod, inp, out, nm=nama: peta.__setitem__(nm, out.detach().cpu())))
with torch.no_grad():
    terbaik.eval(); terbaik(contoh)
for k in kait:
    k.remove()

fig, axes = plt.subplots(2, 8, figsize=(11, 3))
for baris, (nama, fm) in enumerate(peta.items()):
    for kol in range(8):
        axes[baris, kol].imshow(fm[0, kol], cmap='viridis')
        axes[baris, kol].axis('off')
    axes[baris, 0].set_title(f'blok {nama}', loc='left', fontsize=9)
fig.suptitle('Delapan feature map pertama pada dua blok convolution')
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

@torch.no_grad()
def prediksi(model, ds):
    model.eval()
    p, t = [], []
    for xb, yb in loader(ds, 512, False):
        p.append(model(xb.to(DEVICE)).argmax(1).cpu()); t.append(yb)
    return torch.cat(p), torch.cat(t)

pred, target = prediksi(terbaik, ds_val)
fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay.from_predictions(target, pred, display_labels=KELAS,
                                        xticks_rotation=45, colorbar=False, ax=ax)
plt.tight_layout(); plt.show()

salah = (pred != target).nonzero().flatten()[:5]
print('lima contoh yang salah diprediksi:')
for i in salah:
    print(f'  indeks {i.item():>4}: benar={KELAS[target[i]]:>12}  '
          f'diprediksi={KELAS[pred[i]]}')

## Exit ticket

1. Mengapa jumlah parameter convolution tidak berubah ketika ukuran citra diperbesar?
2. Berapa persen parameter CNN baseline yang berada pada layer fully connected pertama?
3. CNN dan FNN memakai jumlah parameter yang hampir sama — lalu apa yang menjelaskan selisih akurasinya?

**Tugas setelah sesi:** kerjakan `starter-mahasiswa.ipynb` — tabel shape manual, empat run terkendali, dua varian, feature map, dan analisis lima kesalahan.